# Reasoning Scaling Laws

Training and benchmark pipeline testing H1 (data efficiency vs. model size) and H2 (agentic compute vs. model scale) on GSM8K.

**Run order:** Cell 0 (env vars) → Cell 1 (install, then **restart kernel**) → Cell 2 (verify imports) → Cell 3 (download from GitHub) → Cell 4 (mount Drive) → Cell 5 (GPU check) → Cell 6 (config) → then either the Test Run cell, or Train 3B / Train 7B → Verify → Benchmark → Results.

**Important:** after running the install cell you must restart the kernel (Runtime → Restart session) before continuing — numpy/torch have C extensions that cannot reload in a running kernel. Re-run Cell 0 after the restart so the env vars are set again.

The 3B and 7B training cells can be run in separate Colab sessions — checkpoints are saved to Drive after every epoch.

In [1]:
# Cell 0 — Environment setup. MUST run before any unsloth import.
import os
print('Env cell ran.')


Env set: PYTORCH_ALLOC_CONF=expandable_segments:False, UNSLOTH_VLLM_STANDBY=1


In [2]:
# Cell 1 — Install dependencies
!pip install -q --upgrade --no-cache-dir --force-reinstall unsloth unsloth_zoo
!pip install -q datasets matplotlib

# >>> After this finishes, RESTART THE KERNEL, then re-run Cell 0, then run Cell 2. <<<


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 277.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 348.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 190.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 270.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 273.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 286.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 349.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 236.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 270.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 247.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 254.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━

In [2]:
import unsloth, trl, transformers, torch
print("unsloth", unsloth.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.5.6
trl 0.24.0
transformers 4.57.6
torch 2.10.0+cu128
vllm 0.19.1


In [ ]:
%%bash
# Replace your-username and your-repo-name with your actual GitHub details
BRANCH="runBranch"
USER="huineith"
REPO="AgenticAndRL_LocalMathModels"

# Define the files to download
FILES=("utils.py" "prompts.py" "rewards.py" "grpo_trainer.py" "agent.py" "benchmarker.py" "warmup_data.json" "start_grpo_sub_process.py")

echo "Fetching fresh files from GitHub and overwriting local copies..."
for file in "${FILES[@]}"; do
    wget -q -O "/content/$file" "https://raw.githubusercontent.com/$USER/$REPO/$BRANCH/$file"
    echo " ✓ Updated: $file"
done

Fetching fresh files from GitHub and overwriting local copies...
 ✓ Updated: utils.py
 ✓ Updated: prompts.py
 ✓ Updated: rewards.py
 ✓ Updated: grpo_trainer.py
 ✓ Updated: agent.py
 ✓ Updated: benchmarker.py
 ✓ Updated: warmup_data.json


In [4]:
# Cell 2 — Mount Google Drive (needed for checkpoint and result saving)
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

CONTENT_DIR = '/content'
os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f'Working directory: {os.getcwd()}')
print('Source files were downloaded from GitHub in the previous cell.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content
Source files were downloaded from GitHub in the previous cell.


In [5]:
# Cell 3 — GPU diagnostics
import torch

if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f'GPU   : {name}')
    print(f'VRAM  : {total:.1f} GB total  |  {free:.1f} GB free')
    print(f'BF16  : {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Switch runtime to L4 GPU in Runtime > Change runtime type.')

GPU   : Tesla T4
VRAM  : 15.6 GB total  |  15.6 GB free
BF16  : False


In [6]:
# Cell 4 — Configuration
MODEL_3B        = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MODEL_7B        = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
DATA_FRACTIONS  = [0.10, 0.20, 0.40]
WARMUP_PATH     = 'warmup_data.json'
SEED            = 42
MAX_EPOCHS      = 3

DRIVE_SAVE_DIR  = '/content/drive/MyDrive/ExamensArbete/checkpoints'
BENCHMARK_DIR   = '/content/drive/MyDrive/ExamensArbete/benchmark_results'

print('Config:')
print(f'  3B model       : {MODEL_3B}')
print(f'  7B model       : {MODEL_7B}')
print(f'  Data fractions : {[int(f*100) for f in DATA_FRACTIONS]}%')
print(f'  Checkpoint dir : {DRIVE_SAVE_DIR}')

Config:
  3B model       : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  7B model       : unsloth/Qwen2.5-7B-Instruct-bnb-4bit
  Data fractions : [10, 20, 40]%
  Checkpoint dir : /content/drive/MyDrive/ExamensArbete/checkpoints


## Test Run (optional)

Run the cell below **instead of Cells 5–8** to do a fast end-to-end check of the whole pipeline.
It trains the 3B model on 1% of the data (≈74 examples, 1 epoch) then runs inference on 20 questions.
Total time on an L4 should be under 15 minutes.

If this cell completes without errors the full pipeline is safe to run.

In [7]:
# Test Cell — end-to-end pipeline check
# Trains 3B on 1% data then runs a 20-question mini benchmark.
# Run Cells 1-4 first so that dependencies are installed and config variables are set.

import gc
import os
import torch
from grpo_trainer import run_grpo
from benchmarker import (
    load_benchmark_questions,
    _load_trained_model,
    _run_nonagentic,
    _format_trained_prompt,
    _compute_metrics,
)
from utils import extract_tagged_answer

TEST_FRACTION    = 0.01   # ~74 training examples
TEST_CHECKPOINT  = f'{DRIVE_SAVE_DIR}/grpo_3b_1pct_best'
TEST_N_QUESTIONS = 20

# ── Step 1: Train ──────────────────────────────────────────────────────────────
print('=' * 60)
print('STEP 1: Training 3B on 1% data (1 epoch max)')
print('=' * 60)

if os.path.exists(TEST_CHECKPOINT):
    print(f'Checkpoint already exists at {TEST_CHECKPOINT} — skipping training.')
else:
    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=TEST_FRACTION,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=1,   # single epoch keeps the test fast
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nTraining complete. Best val accuracy: {best_acc:.4f}')

# ── Step 2: Mini benchmark (non-agentic, 20 questions) ─────────────────────────
print()
print('=' * 60)
print(f'STEP 2: Mini benchmark ({TEST_N_QUESTIONS} questions, 3B no agent)')
print('=' * 60)

questions, solutions = load_benchmark_questions(TEST_N_QUESTIONS, seed=SEED)
model, tokenizer = _load_trained_model(TEST_CHECKPOINT)
results = _run_nonagentic(
    model, tokenizer, questions, solutions,
    _format_trained_prompt, extract_tagged_answer,
)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics = _compute_metrics(results)
print(f'\nTest results ({TEST_N_QUESTIONS} questions):')
print(f'  Accuracy   : {metrics["accuracy"]:.2%}  ({metrics["correct"]}/{metrics["total"]})')
print(f'  Avg tokens : {metrics["avg_tokens"]:.0f}')
print()
print('✓ Pipeline completed without errors.')
print('  Note: accuracy will be low at 1% data — that is expected.')
print('  A non-zero accuracy (> 0) means the reward signal is working.')

STEP 1: Training 3B on 1% data (1 epoch max)
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model: 3b  |  Training examples: 74 (1%)
Validation examples: 100


Map:   0%|          | 0/25 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


...

Step,Training Loss
5,2.787400


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
SFT warmup complete -> /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_3b_1pct_warmup


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 74 | Num Epochs = 1 | Total steps = 74
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / compute_reward / mean,rewards / compute_reward / std
10,0.002600,0.877083,0.495139,84.100000,54.700000,117.900000,0.000000,84.100000,54.700000,117.900000,0.066218,0.877083,0.495139
20,0.002700,0.850000,0.700814,84.575000,63.600000,113.300000,0.000000,84.575000,63.600000,113.300000,0.068303,0.850000,0.700814
30,0.002400,0.937500,0.665713,87.200000,60.800000,116.400000,0.000000,87.200000,60.800000,116.400000,0.058768,0.937500,0.665713
40,0.001900,0.602083,0.737159,88.225000,59.600000,126.300000,0.000000,88.225000,59.600000,126.300000,0.046961,0.602083,0.737159
50,0.002400,0.787500,0.646968,91.825000,62.600000,130.300000,0.000000,91.825000,62.600000,130.300000,0.059835,0.787500,0.646968
60,0.002200,0.877083,0.720726,89.600000,61.100000,138.700000,0.000000,89.600000,61.100000,138.700000,0.054892,0.877083,0.720726
70,0.001800,1.009583,0.819241,85.075000,65.200000,115.500000,0.000000,85.075000,65.200000,115.500000,0.045339,1.009583,0.819241


[Epoch 1] Val accuracy: 0.4400  Best: 0.0000
  -> New best saved to /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_3b_1pct_best
GRPO complete. Best val accuracy: 0.4400
Best checkpoint: /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_3b_1pct_best

Training complete. Best val accuracy: 0.4400

STEP 2: Mini benchmark (20 questions, 3B no agent)
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Test results (20 questions):
  Accuracy   : 45.00%  (9/20)
  Avg tokens : 339

✓ Pipeline completed without errors.
  Note: accuracy will be low at 1% data — that is expected.


## Training

Cells 5 and 6 each run 3 independent GRPO training runs (one per data fraction).
Each run: SFT warmup (1 epoch) → GRPO with early stopping (max 3 epochs).
Checkpoints are saved to Drive after every epoch — safe to resume if the session disconnects.
If a `_best` checkpoint already exists for a run, that run is skipped automatically.

In [ ]:
import os
import time
import subprocess

RUNS = [
    (MODEL_3B, 0.10), (MODEL_3B, 0.20), (MODEL_3B, 0.40),
]

for model_name, frac in RUNS:
    size_tag = "3b" if "3B" in model_name else "7b"
    pct = int(frac * 100)
    expected = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct}pct_best"

    if os.path.exists(expected):
        print(f"SKIP  {size_tag} {pct}% — checkpoint finns redan")
        continue

    print(f"\n{'='*60}\nSTART {size_tag} {pct}%\n{'='*60}", flush=True)
    result = subprocess.run(
        ["python", "start_grpo_sub_process.py",
         "--model", model_name,
         "--fraction", str(frac),
         "--save_dir", DRIVE_SAVE_DIR,
         "--max_epochs", str(MAX_EPOCHS),
         "--seed", str(SEED)],
    )
    print(f"END   {size_tag} {pct}%  (returncode {result.returncode})", flush=True)

    if result.returncode != 0:
        print(f"  VARNING: körningen avslutades med fel — fortsätter till nästa.", flush=True)

    # Ge GPU-minnet tid att frigöras helt innan nästa subprocess startar.
    time.sleep(15)

print("\nAlla körningar klara (eller överhoppade).")

In [ ]:
import os
import time
import subprocess

RUNS = [
    (MODEL_7B, 0.10), (MODEL_7B, 0.20), (MODEL_7B, 0.40),
]

for model_name, frac in RUNS:
    size_tag = "3b" if "3B" in model_name else "7b"
    pct = int(frac * 100)
    expected = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct}pct_best"

    if os.path.exists(expected):
        print(f"SKIP  {size_tag} {pct}% — checkpoint finns redan")
        continue

    print(f"\n{'='*60}\nSTART {size_tag} {pct}%\n{'='*60}", flush=True)
    result = subprocess.run(
        ["python", "start_grpo_sub_process.py",
         "--model", model_name,
         "--fraction", str(frac),
         "--save_dir", DRIVE_SAVE_DIR,
         "--max_epochs", str(MAX_EPOCHS),
         "--seed", str(SEED)],
    )
    print(f"END   {size_tag} {pct}%  (returncode {result.returncode})", flush=True)

    if result.returncode != 0:
        print(f"  VARNING: körningen avslutades med fel — fortsätter till nästa.", flush=True)

    # Ge GPU-minnet tid att frigöras helt innan nästa subprocess startar.
    time.sleep(15)

print("\nAlla körningar klara (eller överhoppade).")

## Benchmark

Uses the 40% checkpoint for Groups 1-4. Group 5 is the zero-shot Qwen2.5-14B-Instruct baseline.
500 questions from the official GSM8K test split (fixed seed = 42).

In [ ]:
# Cell 7 — Verify all required checkpoints exist
import os

to_check = {
    '3B 10%': f'{DRIVE_SAVE_DIR}/grpo_3b_10pct_best',
    '3B 20%': f'{DRIVE_SAVE_DIR}/grpo_3b_20pct_best',
    '3B 40%': f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best',
    '7B 10%': f'{DRIVE_SAVE_DIR}/grpo_7b_10pct_best',
    '7B 20%': f'{DRIVE_SAVE_DIR}/grpo_7b_20pct_best',
    '7B 40%': f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best',
}

all_ok = True
for label, path in to_check.items():
    status = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}]  {label}  ->  {path}')
    if 'MISSING' in status:
        all_ok = False

print()
if all_ok:
    print('All checkpoints present. Ready to run benchmark.')
else:
    print('Some checkpoints are missing. Complete training before running the benchmark.')

In [ ]:
# Cell 8 — Run benchmark (all 5 groups, 500 questions)
from benchmarker import run_benchmark

CHECKPOINT_3B = f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best'
CHECKPOINT_7B = f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best'

summaries = run_benchmark(
    checkpoint_3b=CHECKPOINT_3B,
    checkpoint_7b=CHECKPOINT_7B,
    save_dir=BENCHMARK_DIR,
    n_questions=500,
    seed=SEED,
)

In [ ]:
# Cell 9 — Display results table and plot
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{BENCHMARK_DIR}/benchmark_summary.csv')
df['accuracy_%']         = (df['accuracy'] * 100).round(2)
df['avg_tokens']         = df['avg_tokens'].round(1)
df['tokens_per_correct'] = df['tokens_per_correct'].round(1)

display(df[['group_name', 'accuracy_%', 'avg_tokens', 'tokens_per_correct', 'correct', 'total']])

print()
display(Image(filename=f'{BENCHMARK_DIR}/benchmark_plot.png'))